# CP2K Γ-supercell band unfolding prototype

This notebook tests the CP2K/AiiDA unfolding pipeline with an editable primitive-cell widget.

The notebook now does the following:

1. read coordinates and CP2K cell information;
2. guess a primitive cell from the atomic geometry;
3. expose the guessed primitive/supercell vectors in widgets so the user can correct them;
4. read CP2K `.wfn` with `cp2k-spm-tools`;
5. parse the CP2K overlap matrix;
6. build the primitive/supercell AO mapping;
7. compute unfolded weights using the full non-orthogonal projector;
8. plot black points whose marker size is proportional to unfolded spectral weight.

For now this notebook is focused on 1D/2D systems. For graphene, the standard path is Γ-K-M-Γ.


In [ ]:
# User settings

from pathlib import Path
import numpy as np

wfn_file = Path("aiida-RESTART.wfn")
overlap_file = Path("overlap_matrix.out")
xyz_file = Path("aiida.coords.xyz")
cp2k_input_file = Path("aiida.inp")
cp2k_output_file = Path("aiida.out")

# Energy window used to load KS orbitals from the WFN file.
# cp2k-spm-tools interprets this relative to its reference energy.
orbital_emin_ev = -3.0
orbital_emax_ev =  3.0

# 1 or 2 for now. If None, the notebook guesses from the cell/coordinates.
requested_dim = 2

# Initial lattice-type guess for the standard k-path.
# Choose "auto" to let the notebook infer square/rectangular/hexagonal/oblique in 2D.
default_lattice_type = "auto"

spin = 0


In [ ]:
from cp2k_spm_tools.cp2k_unfolding import (
    build_modulo_lattice_ao_mapping,
    create_primitive_cell_widgets,
    folded_kpoints_from_supercell_matrix,
    guess_2d_lattice_type,
    infer_aos_per_symbol_from_wfn,
    integer_supercell_matrix,
    kfrac_to_cart,
    mo_norms_sparse,
    plot_unfolded_kpath,
    print_eigenvalue_summary,
    project_kpoints_to_kpath,
    read_cp2k_wfn,
    read_primitive_cell_widgets,
    parse_cp2k_overlap_matrix_log_data,
    standard_kpath,
    spectral_weight_full,
    unfold_band_weights_full,
)


## 1. Coordinates, automatic primitive-cell guess, and editable widget

The notebook guesses the primitive vectors from the coordinates and the CP2K supercell, then exposes the guess in editable widgets. Correct the primitive vectors here if needed before running the next cells.


In [ ]:
from IPython.display import display

cell_widgets = create_primitive_cell_widgets(
    xyz_file=xyz_file,
    cp2k_input_file=cp2k_input_file,
    requested_dim=requested_dim,
    default_lattice_type=default_lattice_type,
    tol=2e-3,
    min_quality=0.80,
)

symbols = cell_widgets.symbols
coords = cell_widgets.coords
dim = cell_widgets.dim
primitive_vectors_widget = cell_widgets.primitive_vectors_widget
supercell_vectors_widget = cell_widgets.supercell_vectors_widget
lattice_type_widget = cell_widgets.lattice_type_widget

print("Number of atoms:", len(symbols))
print("Guessed dimensionality:", dim)
print("Guessed primitive vectors [Angstrom]:")
print(cell_widgets.primitive_guess)
print("Guessed supercell vectors [Angstrom]:")
print(cell_widgets.supercell_guess)
print("Guessed lattice type:", cell_widgets.lattice_type_guess)
print("\nEdit the widgets below if needed, then run the next cell.")

display(primitive_vectors_widget, supercell_vectors_widget, lattice_type_widget)


In [ ]:
primitive_vectors, supercell_vectors, lattice_type = read_primitive_cell_widgets(cell_widgets)

print("Using primitive vectors [Angstrom]:")
print(primitive_vectors)
print("Using supercell vectors [Angstrom]:")
print(supercell_vectors)
print("Using lattice type:", lattice_type)

M_check = integer_supercell_matrix(primitive_vectors, supercell_vectors)
print("supercell integer matrix M, defined by S = A @ M:")
print(M_check)
print("det(M):", int(round(abs(np.linalg.det(M_check)))))


## 2. Read WFN, eigenvalues, and coefficients

In [ ]:
wfn = read_cp2k_wfn(
    wfn_file,
    emin=orbital_emin_ev,
    emax=orbital_emax_ev,
)

C = wfn.coeffs[spin]

print("selected eigenvalues:", wfn.evals_ev[spin].shape)
print("coefficient array:", C.shape)
print("reference energy [eV]:", wfn.ref_energy_ev)

print_eigenvalue_summary(wfn, spin=spin, n=10)


## 3. Parse overlap matrix and check MO normalization

In [ ]:
overlap = parse_cp2k_overlap_matrix_log_data(overlap_file, n_atomic_orbitals=C.shape[1])
S = overlap.matrix

print("S shape:", S.shape)
print("S nnz:", S.nnz)
print("first 10 diagonal elements:", S.diagonal()[:10])
print("symmetric pattern:", (S - S.T).nnz == 0)
print("max asymmetry:", abs(S - S.T).max())

norms_S = mo_norms_sparse(C, S)
norms_euclidean = np.einsum("ni,ni->n", C, C)

print("MO norms C S C:")
print(norms_S)
print("Euclidean norms C C:")
print(norms_euclidean)


## 4. Build geometry-based primitive/supercell AO mapping

In [ ]:
aos_per_symbol = infer_aos_per_symbol_from_wfn(symbols, C.shape[1])
print("AO count per symbol inferred from WFN:", aos_per_symbol)

mapping = build_modulo_lattice_ao_mapping(
    symbols=symbols,
    coords_cart=coords,
    primitive_vectors=primitive_vectors,
    supercell_vectors=supercell_vectors,
    aos_per_symbol=aos_per_symbol,
    tol=1e-5,
)

expected_nrep = len(symbols) // len(mapping.basis_frac_coords)
expected_nao_super = C.shape[1]

print("supercell integer matrix M, defined by S = A @ M:")
print(mapping.supercell_integer_matrix)
print("det(M):", round(abs(np.linalg.det(mapping.supercell_integer_matrix))))
print()
print("number of atoms:", len(symbols))
print("nao supercell:", mapping.nao_super)
print("nao primitive:", mapping.nao_prim)
print("number of primitive replicas:", mapping.nrep)
print("expected replicas from atom count:", expected_nrep)
print()
print("basis fractional coordinates modulo primitive lattice:")
print(mapping.basis_frac_coords)
print("atom -> primitive basis atom:")
print(mapping.atom_to_basis)
print("atom -> replica:")
print(mapping.atom_to_replica)
print("replica vectors [Angstrom]:")
print(mapping.replica_vectors_cart)

assert mapping.nrep == expected_nrep, "The number of replicas is not consistent with the atom count."
assert mapping.nao_super == expected_nao_super, "AO mapping does not match WFN coefficient dimension."


## 5. Γ-only sanity check with the full projector

This uses the non-orthogonal projector with the full k-dependent overlap metric `S(k)`.

In [ ]:
k_gamma = np.array([0.0, 0.0, 0.0])

weights_gamma = np.array([
    spectral_weight_full(C[imo], k_gamma, S, mapping)
    for imo in range(C.shape[0])
])

print("full-projector Γ weights:")
print(weights_gamma)


## 6. Folded primitive k-points compatible with the Γ supercell

In [ ]:
k_frac_folded = folded_kpoints_from_supercell_matrix(mapping.supercell_integer_matrix)
k_cart_folded = kfrac_to_cart(k_frac_folded, primitive_vectors)

print("folded k-points in primitive reciprocal fractional coordinates:")
print(k_frac_folded)
print("number of folded k-points:", len(k_frac_folded))
print()
print("folded k-points in Cartesian reciprocal coordinates [1/Angstrom]:")
print(k_cart_folded)


## 7. Full unfolded weights on all folded k-points

For the Γ-only supercell, these are the primitive-cell k-points folded into supercell Γ.

In [ ]:
weights_folded = unfold_band_weights_full(C, k_cart_folded, S, mapping)

np.set_printoptions(precision=4, suppress=True)
print("weights_folded shape:", weights_folded.shape)
print("weights folded [ik, imo]:")
print(weights_folded)
print()
print("sum over folded k-points per MO:")
print(weights_folded.sum(axis=0))
print()
print("dominant folded k-point per MO:")
for imo in range(C.shape[0]):
    ik = int(np.argmax(weights_folded[:, imo]))
    print(
        f"MO {imo:3d}: ik={ik:2d}, "
        f"k_frac={k_frac_folded[ik]}, "
        f"weight={weights_folded[ik, imo]:.6f}, "
        f"E-ref={wfn.evals_ev[spin][imo] - wfn.ref_energy_ev:.6f} eV"
    )


## 8. Standard high-symmetry k-path and unfolded points on that path

The plot uses one color only. Marker size is proportional to the unfolded spectral weight.

For graphene, the default path is Γ-K-M-Γ.

In [ ]:
import matplotlib.pyplot as plt

# Build a standard path for the primitive lattice.
dim = primitive_vectors.shape[0]
hs_points, hs_path = standard_kpath(dim, lattice_type=lattice_type, primitive_vectors=primitive_vectors)

print("lattice type:", lattice_type if lattice_type is not None else guess_2d_lattice_type(primitive_vectors))
print("high-symmetry points [fractional reciprocal coordinates]:")
for label, value in hs_points.items():
    print(f"  {label}: {value}")
print("path:", "-".join(hs_path))

# Select only the folded k-points that lie on the chosen path.
path_k_indices, path_x, path_segments, path_t, path_q_equiv, x_ticks = project_kpoints_to_kpath(
    k_frac_folded,
    hs_points,
    hs_path,
    primitive_vectors,
    tol_cart=1e-6,
)

print("folded k-points on this path:", path_k_indices)
print("their x positions:", path_x)
print("their equivalent fractional coordinates on/near the path:")
print(path_q_equiv)

energies_shifted = wfn.evals_ev[spin] - wfn.ref_energy_ev

ax = plot_unfolded_kpath(
    path_k_indices=path_k_indices,
    path_x=path_x,
    x_ticks=x_ticks,
    x_tick_labels=hs_path,
    energies_ev=energies_shifted,
    weights=weights_folded,
)
plt.show()
